In [ ]:
!pip install transformers torch datasets evaluate

# PyPI: [evaluate](https://pypi.org/project/evaluate/)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

# Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [ ]:
sample = dataset["test"][0]["dialogue"]
sample

"Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye"

In [ ]:
label = dataset["test"][0]["summary"]
label

"Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry."

In [ ]:
def generate_summary(input, llm):
  input_prompt = f"""
                Summarize the following conversations.
                {input}

                Summary:
                  """
  input_ids = tokenizer(sample, return_tensors="pt")
  tokenized_output = llm.generate(input_ids["input_ids"], min_length=30, max_length=200)
  output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)

  return output

output = generate_summary(input=sample, llm=model)

print("Sample:")
print(sample)
print("-"*100)
print("Model generated summary:")
print(output)
print("-"*100)
print("Correct Summary:")
print(label)

Sample:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
----------------------------------------------------------------------------------------------------
Model generated summary:
Hannah: Hey, do you have Betty's number? Amanda: Lemme check. Hannah: Ask Larry. Amanda: He called her last time we were at the park together.
----------------------------------------------------------------------------------------------------
Correct Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


# Prepare Our Dataset

In [ ]:
def tokenize_inputs(example):
    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt = "\n\nSummary: "
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example['dialogue']]
    tokenized_prompt = tokenizer(prompt, padding='max_length', truncation=True, return_tensors='pt', max_length=512)
    tokenized_summary = tokenizer(example['summary'], padding='max_length', truncation=True, return_tensors='pt', max_length=512)

    example['input_ids'] = tokenized_prompt['input_ids']
    example['attention_mask'] = tokenized_prompt['attention_mask']
    example['labels'] = tokenized_summary['input_ids']

    return example

tokenizer.pad_token = tokenizer.eos_token
tokenized_datasets = dataset.map(tokenize_inputs, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'dialogue', 'summary'])
tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

In [ ]:
print(tokenized_datasets["train"].shape)
print(tokenized_datasets["validation"].shape)
print(tokenized_datasets["test"].shape)

(148, 3)
(9, 3)
(9, 3)


In [ ]:
# Contains input_ids labels, which are all tokenized text(ids)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 148
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9
    })
})

In [ ]:
from transformers import TrainingArguments, Trainer


training_args = TrainingArguments(
    output_dir="./bart-cnn-samsum-finetuned",  # local directory
    learning_rate=1e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    auto_find_batch_size=True,
    eval_strategy='epoch',
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.098734,0.132439


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22, training_loss=0.1434898240999742, metrics={'train_runtime': 108.8097, 'train_samples_per_second': 1.36, 'train_steps_per_second': 0.202, 'total_flos': 169034158964736.0, 'train_loss': 0.1434898240999742, 'epoch': 1.0})

In [ ]:
trainer.push_to_hub()

In [ ]:
from google.colab import userdata
username = userdata.get('HF_USERNAME')

In [ ]:
from transformers import AutoModelForSeq2SeqLM

new_model = AutoModelForSeq2SeqLM.from_pretrained(f"{username}/bart-cnn-samsum-finetuned")
output = generate_summary(sample, llm=new_model)

print("Sample:")
print(sample)
print("-"*100)
print("Model(fine-tuned) generated summary:")
print(output)
print("-"*100)
print("Correct Summary:")
print(label)

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
----------------------------------------------------------------------------------------------------
Model generated summary:
Amanda can't find Betty's number, she'll ask Larry. He called her last time they were at the park together. Hannah doesn't know Larry well.
----------------------------------------------------------------------------------------------------
Correct Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
